# 단서 분포 (BDD100K 정상 주행) — Colab GPU

음주 판정 기준을 데이터로 정하기 위한 분석이다 (`docs/risk-criteria.md` 3.1). BDD100K 정상 주행 영상에서 차량별 단서 측정값 분포를 뽑는다.

- **런타임 → 런타임 유형 변경 → T4 GPU** 로 바꾸고 시작한다.
- 결과·영상은 Drive(`WORK`)에 저장된다. **세션이 끊기면 1~5번 셀을 다시 실행하고 7번을 다시 실행하면 끝난 영상은 건너뛰고 이어서 한다.**
- 준비물: Drive `WORK/models/`에 `yolov8n-safewatch.onnx`, `ufld_aihub_aug.onnx` (저장소 `data/models/`의 같은 파일), GitHub 읽기 권한 토큰.

In [ ]:
# 1번 셀 — Drive 연결, GPU 확인
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

In [ ]:
# 2번 셀 — 설정
WORK = '/content/drive/MyDrive/safewatch'          # Drive 작업 폴더 (models/ 를 여기 둔다)
REPO = '/content/SafeWatch_AI'                      # 코드는 Colab 로컬에 (매 세션 새로 받음)
N_VIDEOS = 200                                      # 받을 영상 수 — 집계 차량이 500대 미만이면 늘려서 3·6·7번 다시
BDD_DIR = f'{WORK}/bdd100k'                         # 영상 (Drive — 세션이 끊겨도 남음)
OUT = f'{WORK}/cue_dist/bdd'                        # 결과
import os; os.makedirs(f'{WORK}/models', exist_ok=True); print(os.listdir(f'{WORK}/models'))

In [ ]:
# 3번 셀 — 코드 받기 (dev 브랜치). 비공개 저장소라 토큰을 묻는다 — 토큰은 저장하지 않는다
import getpass, os, subprocess
URL = 'github.com/26-2-Paran-SafeWatch/SafeWatch_AI.git'
if not os.path.exists(REPO):
    tok = getpass.getpass('GitHub 토큰 (repo 읽기 권한): ')
    r = subprocess.run(['git', 'clone', '-q', '-b', 'dev', f'https://{tok}@{URL}', REPO], capture_output=True, text=True)
    del tok
    print(r.stderr.replace('@', ' ') if r.returncode else '받음')
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{URL}'])   # 토큰을 설정에 남기지 않음
else:
    print('이미 있음 — 최신으로 받으려면 런타임을 다시 시작하거나 폴더를 지우고 다시 실행')
!git -C {REPO} log --oneline -1

In [ ]:
# 4번 셀 — 패키지. 차선 모델을 GPU로 돌리려면 onnxruntime 대신 onnxruntime-gpu
!pip -q uninstall -y onnxruntime
!pip -q install onnxruntime-gpu "ultralytics>=8.2,<9.0" "lap>=0.5.12" pyyaml
import onnxruntime as ort; print(ort.__version__, ort.get_available_providers())   # CUDAExecutionProvider가 보여야 함

In [ ]:
# 5번 셀 — 모델 파일을 저장소 data/models/로
import shutil, os
os.makedirs(f'{REPO}/data/models', exist_ok=True)
for f in ['yolov8n-safewatch.onnx', 'ufld_aihub_aug.onnx']:
    src = f'{WORK}/models/{f}'
    assert os.path.exists(src), f'Drive에 없음: {src}'
    shutil.copy(src, f'{REPO}/data/models/{f}'); print('복사', f)

In [ ]:
# 6번 셀 — BDD100K 영상 받기 (낮·맑음, val). 처음엔 라벨(190MB)·목차 읽기에 몇 분, 영상 200개 약 15분.
# 이미 받은 영상은 건너뛴다. 공식 사이트의 묶음 링크가 깨져 있어 큰 zip(1.97TB)에서 고른 것만 잘라 받는다
%cd {REPO}
!python scripts/fetch_bdd100k.py --count {N_VIDEOS} --out {BDD_DIR}

In [ ]:
# 7번 셀 — 분석 (영상 하나 40초 · 끝날 때마다 Drive에 저장, 끊기면 다시 실행해 이어서)
# 첫 영상 로그에서 "UFLD ONNX 로드 … CUDAExecutionProvider"가 보이면 차선이 GPU로 도는 것
%cd {REPO}
!python -u -m scripts.cue_distribution --config configs/colab_cue_eval.yaml --source {BDD_DIR} --output {OUT} --snapshots 2>&1 | grep --line-buffered -v "DEBUG" | grep --line-buffered -E "UFLD ONNX|Using ONNX|^\\[|    차량|WARNING|Error|Traceback|저장됨" 

In [ ]:
# 8번 셀 — 결과 보기 (돌리는 도중에도 끝난 영상까지로 요약을 다시 만들 수 있음: --summary-only)
%cd {REPO}
!python -m scripts.cue_distribution --config configs/colab_cue_eval.yaml --source {BDD_DIR} --output {OUT} --snapshots --summary-only > /dev/null 2>&1
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/summary.md', encoding='utf-8').read()))

## 야간 — 낮과 같은 분석을 밤 영상으로

NHTSA 음주 확률 근거가 전부 야간 운전자 기준이라 야간 성능을 따로 확인한다 (`docs/risk-criteria.md` 6.2 주간·야간 분리 측정).
낮과 날씨 조건(맑음)을 같게 두고 시간대만 바꾼다 — val 밤·맑음 영상은 3,274개.
보는 것: ① 판정까지 간 차량 수(차선이 보류되면 줄어든다 = 밤에 놓치는 정도) ② 분포가 낮보다 부풀었는지(박스·차선 흔들림 = 밤 오탐) ③ 상위 캡처.
세션이 새로 시작됐으면 1~5번 셀을 먼저 실행한다.

In [ ]:
# 9번 셀 — 밤 영상 받기 (낮과 결과 폴더가 따로)
N_NIGHT = 100                                       # 밤은 비교용 — 낮보다 적게 시작
BDD_NIGHT_DIR = f'{WORK}/bdd100k_night'
OUT_NIGHT = f'{WORK}/cue_dist/bdd_night'
%cd {REPO}
!python scripts/fetch_bdd100k.py --count {N_NIGHT} --timeofday night --weather clear --out {BDD_NIGHT_DIR}

In [ ]:
# 10번 셀 — 밤 분석 (끊기면 1~5번, 9번 셀의 위 세 줄(변수)만 실행하고 이 셀 다시 실행 — 이어서 함)
%cd {REPO}
!python -u -m scripts.cue_distribution --config configs/colab_cue_eval.yaml --source {BDD_NIGHT_DIR} --output {OUT_NIGHT} --snapshots 2>&1 | grep --line-buffered -v "DEBUG" | grep --line-buffered -E "UFLD ONNX|Using ONNX|^\\[|    차량|WARNING|Error|Traceback|저장됨" 

In [ ]:
# 11번 셀 — 밤 결과 보기
%cd {REPO}
!python -m scripts.cue_distribution --config configs/colab_cue_eval.yaml --source {BDD_NIGHT_DIR} --output {OUT_NIGHT} --snapshots --summary-only > /dev/null 2>&1
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT_NIGHT}/summary.md', encoding='utf-8').read()))

In [ ]:
# 12번 셀 — 낮 vs 밤 비교 (A = 낮, B = 밤). 'A 95% 기준 → B 걸림'이 5%보다 크게 높으면 밤에 측정값이 부푼다(오탐 ↑)
# 판정까지 간 차량 수도 같이 본다 — 영상 수 대비 밤이 크게 적으면 밤에 차선 보류가 많아 놓친다는 뜻
%cd {REPO}
import csv, glob
from IPython.display import Markdown, display
for name, d in (('낮', OUT), ('밤', OUT_NIGHT)):
    n_vid = len(glob.glob(f'{d}/_per_video/*.json'))          # 처리 끝난 영상 수 (차가 0대인 영상 포함)
    rows = list(csv.DictReader(open(f'{d}/tracks.csv', encoding='utf-8')))
    ok = [r for r in rows if float(r['judged_sec']) >= 3]
    print(f'{name}: 영상 {n_vid}개 · 판정 3초 이상 차량 {len(ok)}대 · 영상당 {len(ok) / max(n_vid, 1):.2f}대')
!python -m scripts.cue_distribution --config configs/colab_cue_eval.yaml --compare {OUT}/tracks.csv {OUT_NIGHT}/tracks.csv --output {WORK}/cue_dist/day_vs_night
display(Markdown(open(f'{WORK}/cue_dist/day_vs_night/compare.md', encoding='utf-8').read()))